# PainNAS on Google Colab

This notebook runs nested supervised early-fusion neural architecture search across all 87 BioVid LOSO folds. Inside every outer fold, NAS sees only the other 86 subjects; its winning architecture is reset, fitted on all 86 source subjects, and evaluated on the untouched target subject. It mirrors the operational setup in `main.ipynb`: the repository is cloned into the Colab VM, BioVid is staged from Google Drive to the local SSD, and all durable artifacts are written back to Drive.

Before running, select **Runtime → Change runtime type → GPU**. Place a BioVid archive such as `BioVid.tar.gz` in `/content/drive/MyDrive/PainData`, or provide an already extracted BioVid `PartA` directory in Drive. The Optuna database and completed LOSO folds are resumable after a disconnect.

## 1. Mount Drive and check out the repository

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/hhihn/FewShotPainAdaptation.git"
PROJECT_DIR = Path("/content/FewShotPainAdaptation")
BRANCH_NAME = "painnas"

if not PROJECT_DIR.exists():
    !git clone -b $BRANCH_NAME $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull --ff-only

%cd $PROJECT_DIR
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

assert (PROJECT_DIR / "painnas").is_dir(), PROJECT_DIR
print("Repository:", PROJECT_DIR)
print("Branch:", BRANCH_NAME)

## 2. Install the pinned experiment dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r $PROJECT_DIR/painnas/requirements-colab.txt
!pip -q install pandas matplotlib

## 3. Stage BioVid on the Colab SSD

Reading the full dataset repeatedly from mounted Drive is slow. The preferred path copies a tar archive to `/content` and extracts it under `/content/PainData`. The staging helper is idempotent within one runtime. If no supported archive exists, the cell falls back to an already extracted Drive directory.

In [ ]:
from pathlib import Path

from data_loaders.dataset_staging import stage_predefined_dataset_from_archive

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/PainData")
LOCAL_DATA_DIR = Path("/content/PainData")
# Add a non-standard archive path here if necessary.
EXTRA_ARCHIVE_CANDIDATES = (
    DRIVE_DATA_DIR / "BioVid 2.tar.gz",
)
DRIVE_EXTRACTED_CANDIDATES = (
    DRIVE_DATA_DIR / "BioVid" / "PartA",
    DRIVE_DATA_DIR / "BioVid 2" / "PartA",
    DRIVE_DATA_DIR / "PartA",
)

try:
    BIOVID_ROOT = stage_predefined_dataset_from_archive(
        "biovid_part_a",
        drive_data_dir=DRIVE_DATA_DIR,
        local_data_dir=LOCAL_DATA_DIR,
        local_archive_dir=Path("/content"),
        extra_archive_candidates=EXTRA_ARCHIVE_CANDIDATES,
    )
    DATA_LOCATION = "local Colab SSD"
except FileNotFoundError as archive_error:
    BIOVID_ROOT = next(
        (
            candidate
            for candidate in DRIVE_EXTRACTED_CANDIDATES
            if (candidate / "Train").is_dir()
            and (candidate / "Test").is_dir()
        ),
        None,
    )
    if BIOVID_ROOT is None:
        raise archive_error
    DATA_LOCATION = "mounted Drive (slower fallback)"

DATA_DIR = BIOVID_ROOT
assert (DATA_DIR / "Train").is_dir(), DATA_DIR
assert (DATA_DIR / "Test").is_dir(), DATA_DIR
for modality in ("GSR", "ECG", "EMG"):
    assert (DATA_DIR / "Train" / modality).is_dir(), modality
    assert (DATA_DIR / "Test" / modality).is_dir(), modality

print("BioVid root:", DATA_DIR)
print("Data location:", DATA_LOCATION)

## 4. Verify the GPU and initialize reproducibility

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import platform
import random
import shutil
import numpy as np
import tensorflow as tf

GPUS = tf.config.list_physical_devices("GPU")
assert GPUS, "No TensorFlow GPU is visible. Select a Colab GPU runtime."
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

SEED = 42
tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
free_gib = shutil.disk_usage("/content").free / 1024**3

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Visible GPUs:", GPUS)
print(f"Free local disk: {free_gib:.1f} GiB")
print("Seed:", SEED)

## 5. Configure the NAS and LOSO run

For each outer fold, the defaults run 10 Optuna trials for at most 20 epochs on a deterministic 69-subject/17-subject inner split. The exact Table 2 network is enqueued as trial 0. The winning architecture is then reset and fitted on all 86 source subjects' `Train` samples for its inner-selected best epoch. Change `RUN_NAME` when changing any configuration value; otherwise the manifest deliberately refuses an incompatible resume.

The outer target subject is excluded from architecture selection, normalization, refitting, and epoch selection. Its `Train` samples remain unused and only its `Test` samples are evaluated.

In [ ]:
from datetime import datetime
import json

from painnas.config import NESTED_PROTOCOL_DESCRIPTION, PainNASConfig
from painnas.io import atomic_write_json

RUN_NAME = "run_001"  # Keep stable to resume; change for a new configuration.
OUTPUT_ROOT = Path("/content/drive/MyDrive/PainNAS")
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME = True
VERBOSE = 1
LOSO_START_INDEX = None  # One-based and inclusive.
LOSO_STOP_INDEX = None   # Use e.g. 1 and 30 to run a chunk.
MAX_FOLDS = None         # Debug only; keep None for the full experiment.

CONFIG = PainNASConfig(
    seed=SEED,
    batch_size=40,
    n_trials=10,
    search_max_epochs=20,
    loso_max_epochs=100,  # Legacy global-LOSO setting; nested refit uses best epoch.
    search_patience=8,
    loso_patience=15,     # Legacy global-LOSO setting; unused by nested refit.
    search_validation_subjects=17,
    max_parameters=32_000_000,
    bootstrap_samples=10_000,
)

atomic_write_json(
    OUTPUT_DIR / "notebook_settings.json",
    {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "repository": str(PROJECT_DIR),
        "branch": BRANCH_NAME,
        "data_dir": str(DATA_DIR),
        "output_dir": str(OUTPUT_DIR),
        "config": CONFIG.to_dict(),
        "loso_start_index": LOSO_START_INDEX,
        "loso_stop_index": LOSO_STOP_INDEX,
        "max_folds": MAX_FOLDS,
        "protocol_description": NESTED_PROTOCOL_DESCRIPTION,
    },
)

print(json.dumps(CONFIG.to_dict(), indent=2))
print("Output directory:", OUTPUT_DIR)
print("Protocol:", NESTED_PROTOCOL_DESCRIPTION)

## 6. Load and validate the real BioVid data

Only T0 and T4 are retained and remapped to binary labels. The expected input is 87 subjects, three modalities in GSR/ECG/EMG order, and 1152 time points. This cell also inspects the first LOSO fold before any training begins.

In [ ]:
import pandas as pd

from painnas.data import build_nested_loso_fold_indices, load_biovid_binary

ARRAYS = load_biovid_binary(str(DATA_DIR), CONFIG)
first_subject = int(ARRAYS.unique_subjects[0])
first_fold = build_nested_loso_fold_indices(
    ARRAYS,
    first_subject,
    validation_subjects=CONFIG.search_validation_subjects,
    seed=CONFIG.seed + 100_000,
)

dataset_summary = pd.DataFrame(
    [
        {
            "samples": len(ARRAYS.y),
            "subjects": len(ARRAYS.unique_subjects),
            "sequence_length": ARRAYS.sequence_length,
            "modalities": ARRAYS.num_modalities,
            "t0_samples": int(np.sum(ARRAYS.y == 0)),
            "t4_samples": int(np.sum(ARRAYS.y == 1)),
            "train_samples": int(
                np.sum(ARRAYS.split_codes == ARRAYS.train_split_code)
            ),
            "test_samples": int(
                np.sum(ARRAYS.split_codes == ARRAYS.test_split_code)
            ),
        }
    ]
)
fold_summary = pd.DataFrame(
    [
        {
            "target_subject": ARRAYS.subject_keys[first_subject],
            "source_subjects": len(first_fold.source_subjects),
            "inner_train_subjects": len(first_fold.inner_train_subjects),
            "inner_validation_subjects": len(
                first_fold.inner_validation_subjects
            ),
            "inner_train_samples": len(first_fold.inner_train),
            "inner_validation_samples": len(first_fold.inner_validation),
            "final_train_samples": len(first_fold.final_train),
            "target_test_samples": len(first_fold.test),
        }
    ]
)
display(dataset_summary)
display(fold_summary)
print("Loaded tensor layout [samples, time, modalities]:", ARRAYS.X.shape)
print("CNN batches will be transposed to [batch, 3, 1152, 1].")

## 7. Inspect the exact Table 2 baseline

In [ ]:
from painnas.model import ArchitectureSpec, build_early_fusion_model

baseline_spec = ArchitectureSpec.baseline()
baseline_model = build_early_fusion_model(baseline_spec)
baseline_model.summary()
print(f"Baseline parameters: {baseline_model.count_params():,}")
del baseline_model
tf.keras.backend.clear_session()

## 8. Run or resume nested LOSO NAS

This cell runs one independent Optuna study inside every selected outer fold. It can be rerun after a disconnect: trial state is stored below each `fold_NNN/search/` directory, completed folds are skipped, and an interrupted final refit restarts only that refit from fresh weights. A GPU out-of-memory candidate is recorded as failed without stopping its study.

In [ ]:
from painnas.nested_loso import run_nested_loso_nas

NESTED_RESULT = run_nested_loso_nas(
    ARRAYS,
    CONFIG,
    OUTPUT_DIR / "nested_loso",
    resume=RESUME,
    start_index=LOSO_START_INDEX,
    stop_index=LOSO_STOP_INDEX,
    max_folds=MAX_FOLDS,
    verbose=VERBOSE,
)
display(pd.DataFrame([NESTED_RESULT]))
print("Nested run directory:", OUTPUT_DIR / "nested_loso")

## 9. Inspect per-fold NAS progress and selected architectures

In [ ]:
import matplotlib.pyplot as plt

NESTED_DIR = OUTPUT_DIR / "nested_loso"
trial_frames = []
for trials_path in sorted(NESTED_DIR.glob("folds/fold_*/search/trials.csv")):
    frame = pd.read_csv(trials_path)
    frame.insert(0, "outer_fold", int(trials_path.parents[1].name.split("_")[1]))
    trial_frames.append(frame)
trials = pd.concat(trial_frames, ignore_index=True) if trial_frames else pd.DataFrame()

best_rows = []
for best_path in sorted(NESTED_DIR.glob("folds/fold_*/search/best_architecture.json")):
    payload = json.loads(best_path.read_text(encoding="utf-8"))
    best_rows.append(
        {
            "outer_fold": int(best_path.parents[1].name.split("_")[1]),
            "target_subject": payload["outer_target_subject"],
            "best_trial": payload["best_trial_number"],
            "best_epoch": payload["best_epoch"],
            "validation_macro_f1": payload["best_validation_macro_f1"],
            **payload["architecture"],
        }
    )
best_architectures = pd.DataFrame(best_rows)

if not trials.empty:
    display(trials.tail(30))
    display(
        trials.groupby(["outer_fold", "state"]).size().rename("trials").to_frame()
    )
if not best_architectures.empty:
    display(best_architectures)

    ax = best_architectures.plot(
        x="outer_fold",
        y="validation_macro_f1",
        marker="o",
        figsize=(10, 4),
        title="Winning inner-validation macro-F1 by outer fold",
    )
    ax.set_xlabel("outer LOSO fold")
    ax.set_ylabel("macro-F1")
    ax.set_ylim(0, 1)
    plt.show()

## 10. Audit target-subject isolation

Every completed result records its inner subjects, excluded target samples, fresh optimizer state, selected epoch, and final refit normalization. This cell checks the central leakage constraints directly from those durable artifacts.

In [ ]:
audit_rows = []
for result_path in sorted(NESTED_DIR.glob("folds/fold_*/result.json")):
    payload = json.loads(result_path.read_text(encoding="utf-8"))
    target = payload["target_subject"]
    target_absent = (
        target not in payload["source_subjects"]
        and target not in payload["inner_train_subjects"]
        and target not in payload["inner_validation_subjects"]
    )
    assert target_absent, result_path
    assert payload["optimizer_initial_iterations"] == 0, result_path
    assert payload["refit_epochs"] >= 1, result_path
    assert payload["refit_epochs_ran"] == payload["refit_epochs"], result_path
    audit_rows.append(
        {
            "fold_index": payload["fold_index"],
            "target_subject": target,
            "target_absent_from_source": target_absent,
            "source_subjects": payload["source_subject_count"],
            "inner_train_subjects": payload["inner_train_subject_count"],
            "inner_validation_subjects": payload["inner_validation_subject_count"],
            "target_train_samples_excluded": payload["target_train_samples_excluded"],
            "target_test_samples": payload["target_test_samples"],
            "fresh_optimizer": payload["optimizer_initial_iterations"] == 0,
            "refit_epochs": payload["refit_epochs"],
        }
    )
display(pd.DataFrame(audit_rows))

## 11. Summarize accumulated LOSO results

In [ ]:
fold_metrics_path = NESTED_DIR / "fold_metrics.csv"
summary_path = NESTED_DIR / "summary.json"
architecture_frequencies_path = NESTED_DIR / "architecture_frequencies.csv"

if fold_metrics_path.exists():
    fold_metrics = pd.read_csv(fold_metrics_path)
    display(fold_metrics)
    ax = fold_metrics.plot(
        x="fold_index",
        y=["accuracy", "macro_f1"],
        marker="o",
        figsize=(12, 4),
        title="Accumulated nested-NAS BioVid LOSO metrics",
    )
    ax.set_xlabel("LOSO fold")
    ax.set_ylabel("score")
    ax.set_ylim(0, 1)
    plt.show()
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(summary["metrics"]).T)
    print("Completed folds:", summary["completed_folds"], "/", summary["total_folds"])
if architecture_frequencies_path.exists():
    display(pd.read_csv(architecture_frequencies_path))

print("Durable artifacts:", OUTPUT_DIR)

## 12. Optional runtime cleanup

Run this only after the desired NAS/LOSO work has finished. All durable artifacts already reside in Drive.

In [ ]:
import gc
import logging

try:
    del ARRAYS
except NameError:
    pass
tf.keras.backend.clear_session()
gc.collect()
logging.shutdown()

try:
    from google.colab import runtime
except ImportError:
    print("Cleanup complete; no hosted Colab runtime was detected.")
else:
    print("Cleanup complete; disconnecting and deleting the Colab runtime.")
    runtime.unassign()